<a href="https://www.kaggle.com/code/msaideepak/fake-new-detection-using-nlp-and-bert?scriptVersionId=322101242" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
!pip install -U transformers

In [2]:
import transformers
print(transformers.__version__)

5.9.0


In [4]:

# Fake News Detection using BERT (GPU Support)


import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments
)
from datasets import Dataset


# 1. CHECK GPU


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using Device:", device)


# 2. LOAD DATASETS



fake_df = pd.read_csv("/kaggle/input/datasets/msaideepak/fake-new-detection-dataset/Fake.csv")
true_df = pd.read_csv("/kaggle/input/datasets/msaideepak/fake-new-detection-dataset/True.csv")

# 3. ADD LABELS


fake_df["label"] = 0
true_df["label"] = 1

# 4. COMBINE DATASETS


df = pd.concat([fake_df, true_df], ignore_index=True)

# 5. USE TITLE + TEXT

df["content"] = df["title"] + " " + df["text"]

# Keep only needed columns
df = df[["content", "label"]]

# Remove null values
df.dropna(inplace=True)

print(df.head())

# 6. TRAIN TEST SPLIT

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["content"].tolist(),
    df["label"].tolist(),
    test_size=0.2,
    random_state=42
)


# 7. LOAD BERT TOKENIZER


tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# 8. TOKENIZATION

train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=256
)

test_encodings = tokenizer(
    test_texts,
    truncation=True,
    padding=True,
    max_length=256
)

# 9. CREATE DATASET CLASS


class FakeNewsDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):

        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item["labels"] = torch.tensor(self.labels[idx])

        return item

    def __len__(self):
        return len(self.labels)

train_dataset = FakeNewsDataset(train_encodings, train_labels)
test_dataset = FakeNewsDataset(test_encodings, test_labels)


# 10. LOAD BERT MODEL


model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

model.to(device)


# 11. EVALUATION METRICS


def compute_metrics(pred):

    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc
    }

# 12. TRAINING ARGUMENTS

training_args = TrainingArguments(
    output_dir="./results",

    num_train_epochs=3,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    warmup_steps=500,

    weight_decay=0.01,

    logging_dir="./logs",

    logging_steps=100,

    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,

    fp16=torch.cuda.is_available()
)
# 13. TRAINER


trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset,

    compute_metrics=compute_metrics
)

# 14. TRAIN MODEL

trainer.train()

# 15. EVALUATE MODEL

results = trainer.evaluate()

print("\nEvaluation Results:")
print(results)

# 16. PREDICTION FUNCTION

def predict_news(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    prediction = torch.argmax(outputs.logits, dim=1).item()

    probability = torch.softmax(outputs.logits, dim=1)

    confidence = probability[0][prediction].item()

    if prediction == 0:
        label = "FAKE NEWS"
    else:
        label = "REAL NEWS"

    return label, confidence


# 17. TEST PREDICTION


sample_news = """
Government secretly planning to ban internet tomorrow
"""

label, confidence = predict_news(sample_news)

print("\nPrediction:", label)
print("Confidence:", round(confidence * 100, 2), "%")


# 18. SAVE MODEL


model.save_pretrained("./fake_news_bert_model")
tokenizer.save_pretrained("./fake_news_bert_model")

print("\nModel Saved Successfully")

Using Device: cuda


/tmp/ipykernel_660/2035918.py:35: DtypeWarning: Columns (4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171) have mixed types. Specify dtype option on import or set low_memory=False.
  fake_df = pd.read_csv("/kaggle/input/datasets/msaideepak/fake-new-detection-dataset/Fake.csv")


                                             content  label
0   Donald Trump Sends Out Embarrassing New Year’...      0
1   Drunk Bragging Trump Staffer Started Russian ...      0
2   Sheriff David Clarke Becomes An Internet Joke...      0
3   Trump Is So Obsessed He Even Has Obama’s Name...      0
4   Pope Francis Just Called Out Donald Trump Dur...      0


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `loggin

...

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

Training Loss,Validation Loss,Epoch,Accuracy
0.000023,0.003585,3,0.999777



Evaluation Results:
{'eval_loss': 0.0035848740953952074, 'eval_accuracy': 0.9997773820124666}

Prediction: FAKE NEWS
Confidence: 100.0 %


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model Saved Successfully


In [5]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

predictions = trainer.predict(test_dataset)

preds = np.argmax(predictions.predictions, axis=1)

print("Accuracy :", accuracy_score(test_labels, preds))
print("Precision:", precision_score(test_labels, preds))
print("Recall   :", recall_score(test_labels, preds))
print("F1 Score :", f1_score(test_labels, preds))

print("\nClassification Report:")
print(classification_report(test_labels, preds))

print("\nConfusion Matrix:")
print(confusion_matrix(test_labels, preds))

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Accuracy : 0.9997773820124666
Precision: 0.9997645396750647
Recall   : 0.9997645396750647
F1 Score : 0.9997645396750647

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      4737
           1       1.00      1.00      1.00      4247

    accuracy                           1.00      8984
   macro avg       1.00      1.00      1.00      8984
weighted avg       1.00      1.00      1.00      8984


Confusion Matrix:
[[4736    1]
 [   1 4246]]
